# Sprint 2 — Working with Text Data
Notebook didático para acompanhar o pipeline implementado em `src/`.


In [ ]:
from pathlib import Path
import sys
import torch

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.tokenization import tokenize_text, build_vocabulary, create_simple_tokenizer
from src.data import create_dataloader_v1
from src.embeddings import TokenAndPositionEmbedding

text = (ROOT / 'data' / 'texto_teste.txt').read_text(encoding='utf-8')
torch.manual_seed(123)


## 1. Tokenização


In [ ]:
frase = 'Um modelo de linguagem aprende padrões.'
tokens = tokenize_text(frase)
print(tokens)
print('Quantidade de tokens:', len(tokens))


## 2. Vocabulário e Token IDs


In [ ]:
corpus_tokens = tokenize_text(text)
str_to_int, int_to_str = build_vocabulary(corpus_tokens)
tokenizer = create_simple_tokenizer(text)
ids = tokenizer.encode(frase)
print('Tamanho do vocabulário:', tokenizer.vocab_size)
print('IDs:', ids)
print('Decode:', tokenizer.decode(ids))


## 3. Sequências de treinamento e DataLoader
O `target` é a sequência de entrada deslocada uma posição para a frente.


In [ ]:
context_length = 8
loader = create_dataloader_v1(text, tokenizer, batch_size=2, max_length=context_length, stride=4, shuffle=False, drop_last=False)
input_batch, target_batch = next(iter(loader))
print('Input:', input_batch.shape)
print('Target:', target_batch.shape)
print(input_batch[0])
print(target_batch[0])


## 4. Token Embeddings + Positional Embeddings


In [ ]:
embedding_dim = 16
embedding_layer = TokenAndPositionEmbedding(tokenizer.vocab_size, embedding_dim, context_length)
embeddings = embedding_layer(input_batch)
print(embeddings.shape)
print('Formato: [batch_size, context_length, embedding_dim]')


## 5. Demonstração do efeito da posição
O mesmo Token ID possui o mesmo token embedding, mas posições distintas recebem positional embeddings diferentes.


In [ ]:
token_id = tokenizer.encode('modelo')[0]
x = torch.tensor([[token_id, token_id]])
layer = TokenAndPositionEmbedding(tokenizer.vocab_size, 8, 2)
token_emb, pos_emb, final_emb = layer.components(x)
print('Token embeddings iguais:', torch.allclose(token_emb[0,0], token_emb[0,1]))
print('Positional embeddings iguais:', torch.allclose(pos_emb[0], pos_emb[1]))
print('Representações finais iguais:', torch.allclose(final_emb[0,0], final_emb[0,1]))


## Saída para a Sprint 3
A entrada do mecanismo de atenção terá o formato **`[batch_size, context_length, embedding_dim]`**.
